<a href="https://colab.research.google.com/github/ritambrakumari/building-llm/blob/main/tokeni.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**STEP 1 TOKENIZATION**

In [1]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    rawtext = f.read()

print(rawtext[:500])  # preview

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it'


In [2]:
import re #for splitting we use re library in pyhon
text="hello world. this is a test"
result=re.split(r'(\s)',text) #split the text whenever whitespace is encountered
__builtins__.print(result) # Use __builtins__.print to avoid conflict

['hello', ' ', 'world.', ' ', 'this', ' ', 'is', ' ', 'a', ' ', 'test']


In [3]:
#splitting comma and period
result=re.split(r'([.,]|\s)',text)
__builtins__.print(result)

['hello', ' ', 'world', '.', '', ' ', 'this', ' ', 'is', ' ', 'a', ' ', 'test']


In [4]:
#removing the redudant whitespace

result=[item for item in result if item.strip()]
__builtins__.print(result)

['hello', 'world', '.', 'this', 'is', 'a', 'test']


In [5]:
import re
# Use the same regex as in the tokenizer and process rawtext
preprocessed = re.split(r'([,.:;?_!"()])|--|\s', rawtext)
preprocessed = [
    item.strip()for item in preprocessed if item is not None and item.strip()
]

In [6]:
print(len(preprocessed))


4422


**STEP 2 CREATING TOKEN IDS**

Token ids are usually assigned alphabetically and uniquely

In [7]:
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)  #contains unique word
print(vocab_size)


1153


In [8]:
vocab= {token:integer for integer, token in enumerate(all_words)} #ennumerates command generally assign number in python

In [9]:
vocab= {token:integer for integer, token in enumerate(all_words)} #ennumerates command generally assign number in python

In [10]:
for i, item in enumerate(vocab.items()):#punctuations are given priority
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
("'Are", 3)
("'It's", 4)
("'coming'", 5)
("'done'", 6)
("'subject", 7)
("'technique'", 8)
("'way", 9)
('(', 10)
(')', 11)
(',', 12)
('.', 13)
(':', 14)
(';', 15)
('?', 16)
('A', 17)
('Ah', 18)
('Among', 19)
('And', 20)
('Arrt', 21)
('As', 22)
('At', 23)
('Be', 24)
('Begin', 25)
('Burlington', 26)
('But', 27)
('By', 28)
('Carlo', 29)
('Chicago', 30)
('Claude', 31)
('Come', 32)
('Croft', 33)
('Destroyed', 34)
('Devonshire', 35)
("Don't", 36)
('Dubarry', 37)
('Emperors', 38)
('Florence', 39)
('For', 40)
('Gallery', 41)
('Gideon', 42)
('Gisburn', 43)
("Gisburn's", 44)
('Gisburns', 45)
('Grafton', 46)
('Greek', 47)
('Grindle', 48)
("Grindle's", 49)
('Grindles', 50)


In [11]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab #token to token id
        self.int_to_str = {i:s for s,i in vocab.items()} #converting token id into token

    def encode(self, text):
        # Use the same regex as vocabulary building, and convert tokens to lowercase
        preprocessed = re.split(r'([,.:;?_!"()])|--|\s', text)
        preprocessed = [
            item.strip()for item in preprocessed if item is not None and item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        print(preprocessed)
        return ids

    def decode(self, ids):
      text = " ".join([self.int_to_str[i] for i in ids])
      text = re.sub(r'\s+([,.:;?!"()])', r'\1', text)
      return text

In [12]:
missing = [token for token in preprocessed if token not in vocab]
print(missing)

[]


Encoding the text


In [13]:
tokenizer=SimpleTokenizerV1(vocab)

text = """It's the last he painted, you know,"
            Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

["It's", 'the', 'last', 'he', 'painted', ',', 'you', 'know', ',', '"', 'Gisburn', 'said', 'with', 'pardonable', 'pride', '.']
[68, 1006, 623, 549, 767, 12, 1147, 617, 12, 1, 43, 870, 1129, 775, 814, 13]


decoding the text


In [14]:
tokenizer.decode(ids)

'It\'s the last he painted, you know," Gisburn said with pardonable pride.'

Dealing with special context token
special context token is used to deal whenever there is no words in tokenizer



In [15]:
#Adding 2 tokens unk and end of text
all_token=sorted(list(set(preprocessed)))
all_token.extend(["<|endoftext|>","<|unk|>"]) #extend add additional token
vocab={token:integer for integer, token in enumerate(all_token)}

In [16]:
len(vocab.items())

1155

In [17]:
for i, item in enumerate(list(vocab.items())[-5:]): #last 5 entries of updated vocabulary
    print(item)


('younger', 1150)
('your', 1151)
('yourself', 1152)
('<|endoftext|>', 1153)
('<|unk|>', 1154)


In [18]:

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [19]:
tokenizer=SimpleTokenizerV2(vocab)
text1="Hello do you like tea?"
text2="in the sunlight terraces of palace"

text="<|endoftext|>".join((text1,text2))
print(text)

Hello do you like tea?<|endoftext|>in the sunlight terraces of palace


In [20]:
tokenizer.encode(text)

[1154, 371, 1147, 650, 994, 16, 1154, 1006, 1154, 1002, 743, 1154]

In [21]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|> do you like tea? <|unk|> the <|unk|> terraces of <|unk|>'

BYTE PAIR ENCODING


In [22]:
!pip3 install tiktoken

In [23]:
import importlib
import tiktoken
print("tiktoken version", importlib.metadata.version("tiktoken"))

tiktoken version 0.13.0


In [24]:
#instantiating the BPE tokenizer from tiktoken
tokenizer=tiktoken.get_encoding("gpt2")

In [25]:
#converting into Token ids "ENCODING"
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [26]:
#converting token ids into "DECODING"
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [27]:
#How BPE tokenizer deals with unkmown token
integers = tokenizer.encode("Akwirw ier")
print(integers)

strings = tokenizer.decode(integers)
print(strings)

[33901, 86, 343, 86, 220, 959]
Akwirw ier


CREATING INPUT-OUTPUT PAIRS

In [28]:
#Implementing Data Loader using Input- Output Pair
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [29]:
enc_sample = enc_text[50:]

In [30]:
#The context size determines how many tokens are included
context_size = 4 #length of the input
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


**Creating Next-Word Prediction**

In [31]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [32]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [33]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [34]:
#why DataLoader?
#because it helps in parallel processing and
#analyze multiple batches at one time..
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [35]:
import torch
print("PyTorch version:", torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)
#Stride means how we are moving

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

PyTorch version: 2.11.0+cpu
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [36]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [37]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
